# Team Toffoli

Our designs, tests and results for the DQC Toffoli challenge. The rules and background are in `Toffoli_Hardware_Hackathon_Challenge.ipynb` in this folder. Run this notebook with the "Python (toffoli)" kernel. Our final case for the judges goes in Section 10 of the starter notebook, and every hardware number in it traces back to a queue ID logged here.

## Decisions

We made these choices before any hardware run, so the hardware data cannot steer them. Each one names the evidence behind it.

### Decision 1: what "best" means

The best design is the one with the highest process fidelity, which measures how close the whole gate, phases included, is to a Toffoli. We measure it with the fidelity range of Section 2:

F_LB = F_T + F_B1 + F_B2 − 2 ≤ process fidelity ≤ min(F_T, F_B1, F_B2)

This needs 24 circuits per design, plus a readout check (8 circuits) that shows how much of the loss comes from readout.

Why this measure and not another:

- A truth table cannot see phases. The starter notebook's lookalike passes every truth-table row but has process fidelity 0.5625, and its range [0.250, 0.625] catches it (Section 2).
- A truth table can also rank designs the wrong way. In the starter kit's noise model, S_TT puts D0 ahead of D1c (0.895 against 0.863), but their true process fidelities are 0.929 and 0.933. The range puts them in the right order on average (Section 3.1), although one run at 128 shots cannot yet separate them (Section 3.3).
- Full process tomography needs 64 inputs × 27 measurement settings = 1,728 circuits, about 110,600 shots at 64 each. Our whole budget is 20,000.
- Standard interleaved randomized benchmarking needs the tested gate to be a Clifford gate, and the Toffoli is not one.

We still report S_TT, since the starter kit uses it, but we never rank on it.

### Decision 2: which designs go to hardware

| Design | What changes against D0 | Question it answers |
|---|---|---|
| D0 | nothing, it is the starter kit's baseline | reference |
| D1c | smaller `rxx` angles: 1.75π in total against 2.25π | do smaller angles make fewer errors? |
| D2 | each `rxx` angle corrected by its own measured error | does calibration help? |

Each comparison with D0 changes one thing. D0c and D1 differ from D0 and D1c by one `rx` or `ry` gate. In the starter kit's model that gate has a 0.1% error, which moves the fidelity by about 0.001, far below our error bars of about 0.02. So they stay on the simulator.

### Decision 3: raw and readout-corrected numbers

- We rank designs on raw F_LB. Every design goes through the same preparation and readout, so a raw difference is a gate difference. Raw F_LB also has the smaller error: 0.015 to 0.020 at 128 shots (Section 3.2), against about 0.03 after readout correction (Section 3.4).
- Next to it we report the readout-corrected range as our estimate of the gate alone. The raw range also contains preparation and readout errors: a perfect gate scores [0.756, 0.912] raw (Section 3.1).

### Decision 4: when two designs differ

Each error bar is 1σ, from √(p(1 − p)/N) per circuit (Section 2) or from the bootstrap for corrected numbers (Section 3.4). Two designs differ only if their gap is larger than 2σ of the difference, √(σ_A² + σ_B²). Otherwise they are tied.

### Hardware shot plan (20,000 in total)

| Step | Shots |
|---|---|
| Readout check, round 1 (8 circuits × 128) | 1,024 |
| Gate probes, round 1 (3 gates × 2 readouts × 256) | 1,536 |
| Round 1: D0, D1c and D2 (3 × 24 circuits × 128) | 9,216 |
| Readout check, round 2 (8 × 128) | 1,024 |
| Gate probes again, round 2 (3 × 2 × 128) | 768 |
| Round 2: the top two designs again (2 × 24 × 128) | 6,144 |
| Spare | 288 |

The readout check runs once in each round. Pooled, it gives 256 shots per state, which the corrected error bars need (Section 3.4), and comparing the two halves shows whether the readout drifted. Round 2 doubles the shots on the two leading designs, which cuts their errors by √2, and runs them close together in time, since hardware drifts. The second gate probe shows whether the `rxx` angles drifted between rounds (it can see a change of about 0.04 rad), which tells us whether D2's correction went stale. The spare 288 shots can rerun two test circuits.

### Other choices

| Choice | What we picked | Why |
|---|---|---|
| Where our work lives | this notebook; the final case goes in Section 10 of the starter notebook | the starter notebook stays as the organizers wrote it, and all our evidence sits in one notebook the judges can rerun |
| Testing order | exact simulation, then noise models, then hardware | hardware shots are limited, and the starter notebook asks for a simulator check first |
| Mystery backend | not used | we have no URL or token for it, so Section 3 uses noise models we build ourselves |
| Shots per test circuit | 128 | 3 designs × 24 circuits × 128 fits round 1 and leaves room for round 2; the F_LB error is then 0.015 to 0.020 (Section 3.2) |
| Error of one row | √(p(1 − p)/N), with (passes + 1)/(N + 2) in place of p | a row with no failures would otherwise get an error of 0 |
| How D2 is built | each of D0's three kinds of `rxx` is measured on its own and corrected by its own error | there is no error model to get wrong, and it worked in every model we tried (Section 3.4) |
| Gate probe | 5 repeats, a Z readout and an X-Y readout, 256 shots each in round 1 and 128 in round 2 | 5 repeats make the error 5 times larger and still leave ±0.63 rad per gate before the angle wraps (Section 3.4) |
| Readout correction | linear inversion without clipping, with bootstrap errors that add 0.5 to every outcome | clipping pulled pass rates down, and the plain bootstrap gave error bars that were too small (Section 3.5) |
| Order in round 1 | readout check and probes at 16:00; if they return within 30 minutes, D0, D1c and D2 go in together in rotated order; if not, D0 and D1c go in at once and D2 follows | designs run close in time give the fairest comparison, but a slow lab must not push round 2 past midnight |
| Readout model | per-qubit matrices, unless the readout check fails the independence test (Section 4.6) at p < 0.05, the 2σ level; then the full 8×8 matrix | fixed before any data, so the gate results cannot steer it; per-qubit has 6 numbers to estimate instead of 56 |
| Who submits | one teammate, from one laptop | the guard against sending a batch twice works within one queue log, so two senders could spend the same shots twice |
| Pair-aware design (D4 in our first plan) | not a separate design | the gate probe already measures and corrects each pair on its own, so D2 covers it |
| Pauli twirling (D3 in our first plan) | not planned | twirling makes each gate's error random, which stops coherent errors from adding up across the five gates. In the disclosed model that lifts D0 from 0.929 to 0.941 (same superoperator method as Section 3), while correcting the angles (D2*) reaches 0.990. It also needs many random versions of each circuit, and each version is its own submission |

Still open: which design we run on IonQ Forte 1 after midnight. We decide that from the lab results.

What we changed along the way, and why, is in Section 3.5.

## 0. Setup

Team values live in `.env` at the repository root, which git ignores. The first cell copies them into the environment and prints only whether each one is set. It runs before the challenge package is imported, because the mystery-backend client reads its URL when it is imported.

Every hardware submission is logged to `submissions/queue_log.json` (also ignored by git), so each number we present traces back to a queue ID.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def load_env(path):
    """Copy KEY=VALUE lines from a .env file into os.environ (values already set win); KEY="value" counts as KEY=value."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, sep, value = line.partition("=")
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
            value = value[1:-1]
        if sep and value:
            os.environ.setdefault(key.strip(), value)


load_env(ROOT / ".env")
for key in ("TEAM_ID", "TEAM_SECRET", "MYSTERY_BACKEND_URL", "MYSTERY_BACKEND_TOKEN"):
    print(f"{key:22s} set: {bool(os.environ.get(key))}")

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, qasm2
from qiskit.circuit.library import SXGate
from qiskit.quantum_info import Operator, Statevector
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGatesDecomposition

from toffoli_challenge import (
    TOFFOLI_INPUTS,
    baseline_native_toffoli,
    compare_to_toffoli,
    validate_submission_circuit,
    with_input_preparation,
)

RESULTS_DIR = ROOT / "returned_results"
QUEUE_LOG = ROOT / "submissions" / "queue_log.json"


def log_submission(receipt, log_path=QUEUE_LOG, **details):
    """Append one hardware submission (its queue ID and what it tested) to the local log."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = json.loads(log_path.read_text()) if log_path.exists() else []
    entry = {"queue_id": receipt["submission"]["queue_id"],
             "logged_at": datetime.now().isoformat(timespec="seconds"), **details}
    log.append(entry)
    log_path.write_text(json.dumps(log, indent=2))
    return entry


print("repository root:", ROOT.name)

## 1. Designs

A Toffoli needs at least five two-qubit gates (Yu, Duan and Ying, *Phys. Rev. A* 88, 010304, 2013), so the baseline's five `rxx` gates cannot be cut. Two things can still change: the size of the `rxx` angles, and the number of noisy one-qubit gates around them. `rz` is done in software and is free, so only `rx`, `ry` and `rxx` cost anything.

D0 is the starter kit's baseline. It is Qiskit's compilation, with five `rxx` and a total angle of 2.25π.

D1 uses the construction from Barenco et al., *Phys. Rev. A* 52, 3457 (1995): a controlled-√X from c2 to t, a CNOT from c1 to c2, a controlled-√X† from c2 to t, the same CNOT again, and a controlled-√X from c1 to t. When both controls are 1, the target gets √X twice, which is X. When exactly one control is 1, the target gets one √X and one √X†, which cancel. A controlled-√X needs only `rxx(π/4)`, half the angle of a CNOT's `rxx(π/2)`, so the total angle is 3 × π/4 + 2 × π/2 = 1.75π.

D2 is not a new circuit. It is D0 with each `rxx` angle changed to cancel the error we measure on that same gate at the start of the hardware session (rehearsed in Section 3.4).

### Building each block from one `rxx`

A controlled-√X is exp(iπ/8 (I − Z)⊗(I − X)). When the control is 0 the exponent is 0. When it is 1 the target gets exp(iπ/4 (I − X)), which is √X.

Multiplying out the exponent leaves one two-qubit term, exp(iπ/8 Z⊗X), plus one-qubit rotations: `rz(π/4)` on the control and `rx(π/4)` on the target. An `ry(π/2)` on the control turns its Z into X, so the two-qubit term becomes exp(iπ/8 X⊗X), which is `rxx(-π/4)`.

A CNOT is exp(iπ/4 (I − Z)⊗(I − X)), so the same steps give `rxx(-π/2)`. The next cell checks each block against Qiskit's own gate.

In [ ]:
C1, C2, T = 2, 1, 0  # challenge wires: q[2] = c1, q[1] = c2, q[0] = t


def controlled_sqrt_x(qc, control, target, inverse=False):
    """Controlled-sqrt(X), exact up to global phase, built from one rxx(-pi/4)."""
    s = -1 if inverse else 1
    qc.ry(np.pi / 2, control)
    qc.rxx(-s * np.pi / 4, control, target)
    qc.ry(-np.pi / 2, control)
    qc.rz(s * np.pi / 4, control)
    qc.rx(s * np.pi / 4, target)


def cnot_native(qc, control, target):
    """CNOT, exact up to global phase, built from one rxx(-pi/2)."""
    qc.ry(np.pi / 2, control)
    qc.rxx(-np.pi / 2, control, target)
    qc.ry(-np.pi / 2, control)
    qc.rz(np.pi / 2, control)
    qc.rx(np.pi / 2, target)


checks = [
    ("controlled-sqrt(X)", lambda qc: controlled_sqrt_x(qc, 1, 0), lambda qc: qc.append(SXGate().control(1), [1, 0])),
    ("controlled-sqrt(X)^dagger", lambda qc: controlled_sqrt_x(qc, 1, 0, inverse=True),
     lambda qc: qc.append(SXGate().inverse().control(1), [1, 0])),
    ("CNOT", lambda qc: cnot_native(qc, 1, 0), lambda qc: qc.cx(1, 0)),
]
for name, build, build_reference in checks:
    block, reference = QuantumCircuit(2), QuantumCircuit(2)
    build(block)
    build_reference(reference)
    print(f"{name:26s} matches Qiskit's gate: {Operator(block).equiv(Operator(reference))}")

### The designs and their costs

Between `rxx` gates, the one-qubit gates on each wire can be merged into one rotation, written rz · ry · rz. Since `rz` is free, each merged run costs at most one `ry`. `compress_1q` does this with Qiskit's `Optimize1qGatesDecomposition` pass and leaves every `rxx` as it is.

In [ ]:
def small_angle_toffoli():
    """D1: the Barenco et al. construction with controls c1 = q[2], c2 = q[1] and target t = q[0]."""
    qc = QuantumCircuit(3)
    controlled_sqrt_x(qc, C2, T)
    cnot_native(qc, C1, C2)
    controlled_sqrt_x(qc, C2, T, inverse=True)
    cnot_native(qc, C1, C2)
    controlled_sqrt_x(qc, C1, T)
    return qc


def compress_1q(circuit):
    """Merge each run of one-qubit gates into rz-ry-rz, so each run costs at most one ry."""
    return PassManager([Optimize1qGatesDecomposition(basis=["rz", "ry"])]).run(circuit)


def costs(circuit):
    """Return (number of rxx, total |rxx angle| / pi, number of rx and ry)."""
    angles = [float(i.operation.params[0]) for i in circuit.data if i.operation.name == "rxx"]
    costly_1q = sum(1 for i in circuit.data if i.operation.name in ("rx", "ry"))
    return len(angles), sum(map(abs, angles)) / np.pi, costly_1q


designs = {
    "D0": baseline_native_toffoli(),
    "D0c": compress_1q(baseline_native_toffoli()),
    "D1": small_angle_toffoli(),
    "D1c": compress_1q(small_angle_toffoli()),
}

print(f"{'design':7s} {'rxx':>4s} {'total angle':>12s} {'rx + ry':>8s} {'ideal process fidelity':>23s}")
for name, circuit in designs.items():
    validate_submission_circuit(circuit)
    n_rxx, total_angle, costly_1q = costs(circuit)
    _, process_fidelity, _ = compare_to_toffoli(circuit)
    print(f"{name:7s} {n_rxx:4d} {total_angle:10.3f}pi {costly_1q:8d} {process_fidelity:23.6f}")

In [ ]:
def rxx_gates(circuit):
    """The distinct rxx gates in a circuit, as (sorted qubit pair, angle)."""
    gates = []
    for instruction in circuit.data:
        if instruction.operation.name == "rxx":
            pair = tuple(sorted(circuit.find_bit(q).index for q in instruction.qubits))
            gate = (pair, round(float(instruction.operation.params[0]), 10))
            if gate not in gates:
                gates.append(gate)
    return gates


def correct_each(circuit, errors):
    """Copy a circuit with each rxx(theta) replaced by rxx(theta - error), using that gate's own error.

    errors maps each (sorted qubit pair, angle) from rxx_gates to how far that gate turns past its angle, in radians.
    """
    out = circuit.copy_empty_like()
    for instruction in circuit.data:
        if instruction.operation.name == "rxx":
            theta = float(instruction.operation.params[0])
            pair = tuple(sorted(circuit.find_bit(q).index for q in instruction.qubits))
            out.rxx(theta - errors[(pair, round(theta, 10))], *instruction.qubits)
        else:
            out.append(instruction)
    return out


print("rxx gates in D0, as (pair, angle / pi):", [(pair, round(theta / np.pi, 3)) for pair, theta in rxx_gates(designs["D0"])])
display(designs["D1c"].draw("mpl", fold=-1))

## 2. Proof method: the fidelity range

A truth table checks only probabilities, so it cannot see the relative phases that a Toffoli must get right (Section 8 of the starter notebook). We use three test sets that together bound the process fidelity from both sides. The method is from Mičuda et al., arXiv:1306.1141, and builds on Hofmann, *Phys. Rev. Lett.* 94, 160504 (2005).

| Set | c1 | c2 | t | What an ideal Toffoli does |
|---|---|---|---|---|
| T | Z | Z | Z | flips t when c1 = c2 = 1 (the truth table) |
| B1 | X | Z | X | flips c1 between + and − when c2 = 1 and t = − |
| B2 | Z | X | X | flips c2 between + and − when c1 = 1 and t = − |

Each set has 8 inputs, and each ideal output is one bitstring, so every shot is a pass or a fail. F_T, F_B1 and F_B2 are the average pass rates, and

F_LB = F_T + F_B1 + F_B2 − 2 ≤ process fidelity ≤ min(F_T, F_B1, F_B2).

Why B1 and B2 see phases: a Toffoli with the target in |−⟩ multiplies the state by −1 when both controls are 1 (phase kickback). On the controls this acts as a CZ. With c2 = 1 that CZ is a Z on c1, which swaps |+⟩ and |−⟩. A circuit with the wrong relative phases gets this swap wrong, even when its truth table is perfect.

The cost is 24 circuits per design: 3,072 shots at 128 shots each.

On hardware, every test circuit also has state preparation and readout, so the range bounds the fidelity of the whole run, not of the gate alone. Readout error by itself takes a perfect gate's range down to [0.756, 0.912] (Section 3.1). So we report the raw range, which needs no correction, and next to it a readout-corrected range as our estimate of the gate alone (Section 3.4).

In [ ]:
TEST_SETS = {"T": ("Z", "Z", "Z"), "B1": ("X", "Z", "X"), "B2": ("Z", "X", "X")}  # bases of (c1, c2, t)
LABELS = {"Z": ("0", "1"), "X": ("+", "-")}
FLIP = {"+": "-", "-": "+"}


def prepare(qc, qubit, label):
    """Prepare |label> from |0> with native gates (table in Section 8 of the starter notebook)."""
    if label == "1":
        qc.rx(np.pi, qubit)
    elif label == "+":
        qc.ry(np.pi / 2, qubit)
    elif label == "-":
        qc.ry(-np.pi / 2, qubit)


def to_z_basis(qc, qubit, basis):
    """Rotate so the lab's Z measurement reads the X basis: |+> gives 0 and |-> gives 1."""
    if basis == "X":
        qc.ry(-np.pi / 2, qubit)


def ideal_output(test_set, c1, c2, t):
    """Output labels of an ideal Toffoli for input labels (c1, c2, t) in one test set."""
    if test_set == "T":
        return c1, c2, str(int(t) ^ (int(c1) & int(c2)))
    if test_set == "B1":
        return (FLIP[c1] if c2 == "1" and t == "-" else c1), c2, t
    return c1, (FLIP[c2] if c1 == "1" and t == "-" else c2), t


def as_bits(labels):
    """The lab's readout string for output labels, in c1 c2 t order."""
    return "".join({"0": "0", "1": "1", "+": "0", "-": "1"}[label] for label in labels)


def test_circuits(candidate):
    """Return {(test set, input labels): (circuit, expected readout)} for all 24 tests."""
    tests = {}
    for name, (b1, b2, bt) in TEST_SETS.items():
        for c1 in LABELS[b1]:
            for c2 in LABELS[b2]:
                for t in LABELS[bt]:
                    qc = QuantumCircuit(3)
                    prepare(qc, C1, c1)
                    prepare(qc, C2, c2)
                    prepare(qc, T, t)
                    qc.compose(candidate, inplace=True)
                    for qubit, basis in ((C1, b1), (C2, b2), (T, bt)):
                        to_z_basis(qc, qubit, basis)
                    validate_submission_circuit(qc)
                    tests[(name, c1 + c2 + t)] = (qc, as_bits(ideal_output(name, c1, c2, t)))
    return tests


print("test circuits per design:", len(test_circuits(designs["D0"])))

### Sign check

The starter notebook warns that signs are easy to get wrong. The next cell checks, with exact simulation, that each preparation gives the right state, and that the X-basis readout sends |+⟩ to 0 and |−⟩ to 1. It also checks that set T expects the same outputs as the starter kit's truth-table helper.

In [ ]:
from toffoli_challenge import expected_output_bitstring

for basis, labels in LABELS.items():
    for label in labels:
        prep = QuantumCircuit(1)
        prepare(prep, 0, label)
        right_state = Statevector(prep).equiv(Statevector.from_label(label))
        to_z_basis(prep, 0, basis)
        p_expected = Statevector(prep).probabilities_dict().get(as_bits(label), 0.0)
        print(f"|{label}>: right state {right_state}, read in {basis} basis as {as_bits(label)} with probability {p_expected:.3f}")

same = all(expected == expected_output_bitstring(key[1])
           for key, (_, expected) in test_circuits(designs["D0"]).items() if key[0] == "T")
print("set T matches the starter kit's truth table:", same)

### From counts to the fidelity range

Each test gives a pass rate p = (shots with the expected readout) / N. Its 1σ error is √(p(1 − p)/N). A row with no failures would get an error of zero, which is too optimistic, so the error uses (passes + 1)/(N + 2) in place of p. The error of an average over 8 rows is √(sum of the row variances) / 8, and the error of F_LB adds the three set errors in quadrature. Readout-corrected numbers get their errors by resampling instead (Section 3.4).

In [ ]:
def fidelity_range(counts_by_test, tests):
    """Pass rates per test set, the fidelity range and S_TT, each with a 1 sigma error.

    counts_by_test maps (test set, input labels) to the counts dict of that circuit.
    """
    rates, errors = {}, {}
    for name in TEST_SETS:
        rows, variances = [], []
        for key, (_, expected) in tests.items():
            if key[0] != name:
                continue
            counts = counts_by_test[key]
            shots = sum(counts.values())
            passes = counts.get(expected, 0)
            rows.append(passes / shots)
            p_err = (passes + 1) / (shots + 2)
            variances.append(p_err * (1 - p_err) / shots)
        rates[name] = float(np.mean(rows))
        errors[name] = float(np.sqrt(np.sum(variances)) / len(rows))
    weakest = min(rates, key=rates.get)
    return {
        "F": rates,
        "F_err": errors,
        "S_TT": rates["T"],
        "S_TT_err": errors["T"],
        "F_LB": rates["T"] + rates["B1"] + rates["B2"] - 2,
        "F_LB_err": float(np.sqrt(sum(e ** 2 for e in errors.values()))),
        "upper": rates[weakest],
        "upper_err": errors[weakest],
    }


def report(name, result):
    f, e = result["F"], result["F_err"]
    print(f"{name:10s} S_TT {f['T']:.3f}±{e['T']:.3f}   F_B1 {f['B1']:.3f}±{e['B1']:.3f}   F_B2 {f['B2']:.3f}±{e['B2']:.3f}   "
          f"range [{result['F_LB']:.3f}±{result['F_LB_err']:.3f}, {result['upper']:.3f}±{result['upper_err']:.3f}]")

### Check on the ideal simulator

D0 and D1c should give 1 for every set. The starter notebook's lookalike (Section 8) has a perfect truth table but the wrong phases, so B1 and B2 should catch it, and its true process fidelity should land inside its range.

In [ ]:
from qiskit import transpile


def lookalike_toffoli():
    """The starter notebook's relative-phase lookalike: perfect truth table, wrong gate."""
    qc = QuantumCircuit(3)
    qc.ry(np.pi / 4, 0)
    qc.cx(1, 0)
    qc.ry(np.pi / 4, 0)
    qc.cx(2, 0)
    qc.ry(-np.pi / 4, 0)
    qc.cx(1, 0)
    qc.ry(-np.pi / 4, 0)
    return transpile(qc, basis_gates=["rx", "ry", "rz", "rxx"], optimization_level=3, seed_transpiler=1)


def exact_counts(circuit, scale=10**6):
    """Noise-free outcome weights, used in place of counts for ideal checks."""
    return {k: v * scale for k, v in Statevector(circuit).probabilities_dict().items() if v > 1e-12}


for name, candidate in {"D0": designs["D0"], "D1c": designs["D1c"], "lookalike": lookalike_toffoli()}.items():
    tests = test_circuits(candidate)
    result = fidelity_range({key: exact_counts(qc) for key, (qc, _) in tests.items()}, tests)
    _, true_fidelity, _ = compare_to_toffoli(candidate)
    report(name, result)
    print(f"{'':10s} true process fidelity {true_fidelity:.4f}")

In [ ]:
# One B1 test for D1c: c1 in |+>, c2 = 1, t in |->, so an ideal Toffoli turns c1 into |->.
example, expected = test_circuits(designs["D1c"])[("B1", "+1-")]
print("expected readout c1 c2 t:", expected)
display(example.draw("mpl", fold=-1))

## 3. Practice runs on simulators

Before spending hardware shots, we run the whole method on noise models and ask four questions. Does each design score as theory predicts? Are the error bars right? Can 128 shots per circuit tell the designs apart? Can the 4 PM calibration steps recover a known error?

| Model | Gate error | Readout error | Why we use it |
|---|---|---|---|
| disclosed | every `rxx` turns 0.2 rad too far, and `rx`/`ry` are 0.1% depolarizing | yes | the starter kit's model (`toffoli_challenge/known_noise.py`) |
| angle | every `rxx` turns 5% too far, plus depolarizing that grows with the angle (2% for a π/2 gate) | yes | a machine where smaller angles really make fewer errors, the case where D1c should win |
| readout only | none | yes | shows how much readout alone costs |
| large offset | every `rxx` turns 0.35 rad too far | yes | tests the gate probe (3.4) past the point where our first probe failed |
| same direction | every `rxx` turns 0.15 rad too far in its own direction, so the error has the sign of the angle | yes | an error that one offset plus one scale factor cannot describe; tests the gate probe (3.4) |

The first three models test the whole method. The last two test only the gate probe.

The readout error is the starter kit's: P(read 1 given 0) = 0.015 and P(read 0 given 1) = 0.040. The "true fidelity" of a design is the exact process fidelity of its noisy gate alone, computed from Aer's superoperator simulation. Real hardware never gives us this number, but the simulator does, so we can check whether our measured range tells the truth.

In [ ]:
from qiskit.quantum_info import SuperOp, process_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, coherent_unitary_error, depolarizing_error

from toffoli_challenge import ideal_toffoli_in_submission_order

TARGET = Operator(ideal_toffoli_in_submission_order())
NATIVE = ["rx", "ry", "rz", "rxx"]
READOUT = [[0.985, 0.015], [0.040, 0.960]]  # rows: prepared 0, 1; columns: read 0, 1


def gate_noise(offset=0.0, depolarizing_1q=0.001):
    """Gate errors only: an optional fixed rxx over-rotation and depolarizing on rx and ry."""
    model = NoiseModel(basis_gates=NATIVE)
    model.add_all_qubit_quantum_error(depolarizing_error(depolarizing_1q, 1), ["rx", "ry"])
    if offset:
        extra = QuantumCircuit(2)
        extra.rxx(offset, 0, 1)
        model.add_all_qubit_quantum_error(coherent_unitary_error(Operator(extra).data), "rxx")
    return model


def with_readout(model):
    model.add_all_qubit_readout_error(ReadoutError(READOUT))
    return model


def angle_noise(circuit, relative=0.05, depolarizing_per_half_pi=0.02):
    """rxx(theta) becomes rxx(theta * (1 + relative)) followed by depolarizing that grows with |theta|."""
    out = circuit.copy_empty_like()
    for instruction in circuit.data:
        if instruction.operation.name == "rxx":
            theta = float(instruction.operation.params[0])
            out.rxx(theta * (1 + relative), *instruction.qubits)
            p = depolarizing_per_half_pi * abs(theta) / (np.pi / 2)
            out.append(depolarizing_error(p, 2).to_instruction(), instruction.qubits)
        else:
            out.append(instruction)
    return out


def same_direction_noise(circuit, size=0.15):
    """rxx(theta) becomes rxx(theta + size * sign(theta)): every rxx turns too far in its own direction."""
    out = circuit.copy_empty_like()
    for instruction in circuit.data:
        if instruction.operation.name == "rxx":
            theta = float(instruction.operation.params[0])
            out.rxx(theta + size * np.sign(theta), *instruction.qubits)
        else:
            out.append(instruction)
    return out


# name: (circuit change, noise model for sampling counts, noise model for the true-fidelity channel)
MODELS = {
    "disclosed": (lambda c: c.copy(), with_readout(gate_noise(offset=0.2)), gate_noise(offset=0.2)),
    "angle": (angle_noise, with_readout(gate_noise()), gate_noise()),
    "readout only": (lambda c: c.copy(), with_readout(NoiseModel(basis_gates=NATIVE)), None),
    "large offset": (lambda c: c.copy(), with_readout(gate_noise(offset=0.35)), gate_noise(offset=0.35)),
    "same direction": (same_direction_noise, with_readout(gate_noise()), gate_noise()),
}
MAIN_MODELS = ["disclosed", "angle", "readout only"]
# how far rxx(theta) turns past theta in each model
TRUE_ERROR = {
    "disclosed": lambda theta: 0.2,
    "angle": lambda theta: 0.05 * theta,
    "readout only": lambda theta: 0.0,
    "large offset": lambda theta: 0.35,
    "same direction": lambda theta: 0.15 * np.sign(theta),
}


def true_fidelity(candidate, model):
    """Exact process fidelity of the noisy gate alone (no preparation or readout error)."""
    change, _, channel_noise = MODELS[model]
    qc = change(candidate.copy())
    qc.save_superop()
    sim = AerSimulator(method="superop", noise_model=channel_noise) if channel_noise else AerSimulator(method="superop")
    return process_fidelity(SuperOp(sim.run(qc).result().data()["superop"]), TARGET)


def simulate(candidate, model, shots=128, seed=1):
    """Run all 24 test circuits on a noise model; return (fidelity_range result, counts, tests)."""
    change, noise, _ = MODELS[model]
    tests = test_circuits(candidate)
    keys = list(tests)
    batch = []
    for key in keys:
        qc = change(tests[key][0])
        qc.measure_all()
        batch.append(qc)
    result = AerSimulator(noise_model=noise, seed_simulator=seed).run(batch, shots=shots).result()
    counts = {key: result.get_counts(i) for i, key in enumerate(keys)}
    return fidelity_range(counts, tests), counts, tests

### 3.1 The full method at 128 shots, repeated 20 times

D2* is D0 with each `rxx` corrected by the model's exact error on that gate (`correct_each`). That is perfect calibration, the best D2 could be. Each design is run with 20 different random seeds. "spread" is the standard deviation of F_LB across those 20 runs, which should match the error bar that `fidelity_range` predicts from a single run.

In [ ]:
d0_gates = rxx_gates(designs["D0"])
perfect_d2 = {model: correct_each(designs["D0"], {gate: TRUE_ERROR[model](gate[1]) for gate in d0_gates}) for model in MAIN_MODELS}
practice = {}
for model in MAIN_MODELS:
    print(f"{model} model")
    for name, candidate in {"D0": designs["D0"], "D1c": designs["D1c"], "D2*": perfect_d2[model]}.items():
        runs = [simulate(candidate, model, seed=s)[0] for s in range(20)]
        lower = np.array([r["F_LB"] for r in runs])
        practice[(model, name)] = lower
        print(f"  {name:4s} true {true_fidelity(candidate, model):.3f} | S_TT {np.mean([r['S_TT'] for r in runs]):.3f} | "
              f"range [{lower.mean():.3f}, {np.mean([r['upper'] for r in runs]):.3f}] | "
              f"F_LB spread {lower.std(ddof=1):.4f}, predicted {np.mean([r['F_LB_err'] for r in runs]):.4f}")

### 3.2 Are the error bars right?

Twenty runs are too few to judge a spread (its own uncertainty is about 16%). The next cell repeats D0 200 times on each model.

In [ ]:
for model in MAIN_MODELS:
    runs = [simulate(designs["D0"], model, seed=s)[0] for s in range(200)]
    print(f"{model:12s} F_LB spread {np.std([r['F_LB'] for r in runs], ddof=1):.4f} vs predicted "
          f"{np.mean([r['F_LB_err'] for r in runs]):.4f} | S_TT spread {np.std([r['S_TT'] for r in runs], ddof=1):.4f} "
          f"vs predicted {np.mean([r['S_TT_err'] for r in runs]):.4f}")

### 3.3 Can 128 shots tell the designs apart?

Decision 4 says two designs differ only if their gap is larger than 2σ of the difference. For one run of each design, σ of the difference is √(σ_A² + σ_B²), using the spreads from 3.1.

In [ ]:
for model in MAIN_MODELS:
    for a, b in (("D1c", "D0"), ("D2*", "D0"), ("D2*", "D1c")):
        gap = practice[(model, a)].mean() - practice[(model, b)].mean()
        two_sigma = 2 * np.sqrt(practice[(model, a)].var(ddof=1) + practice[(model, b)].var(ddof=1))
        verdict = "different" if abs(gap) > two_sigma else "tied"
        print(f"{model:12s} F_LB({a}) - F_LB({b}) = {gap:+.3f}, 2 sigma = {two_sigma:.3f} -> {verdict}")

### 3.4 Rehearsal of the first hardware steps

Two measurements come first on the hardware.

The readout check prepares each of 000 to 111 with `rx(π)` flips and measures straight away (8 circuits × 128 shots, once in each round). It gives each qubit's P(read 1 given 0) and P(read 0 given 1) as a 2×2 matrix. We undo readout error by solving the linear equations those matrices give (linear inversion). The answer can have small negative entries. We keep them, because setting them to 0 pulls every pass rate down (shown below). Correcting also adds the readout check's own shot noise, so the corrected range gets its error bars from a bootstrap: we redraw every circuit's counts and the readout check's counts 200 times from their own measured frequencies, redo the correction each time, and take the spread. Before each redraw we add 0.5 to the count of every one of the 8 outcomes. Without it, an outcome seen 0 times in 128 shots can never be redrawn, and the error bars came out 10 to 15% too small (Section 3.5).

The gate probe measures how far each `rxx` in D0 really turns. D0 has three kinds: `rxx(π/2)` on pairs (0, 1) and (0, 2), and `rxx(-π/4)` on pair (1, 2). For each one we repeat the gate 5 times on |00⟩, which makes its error 5 times larger. The pair ends in cos(Φ/2)|00⟩ − i sin(Φ/2)|11⟩, where Φ is the real total angle. Two readouts give Φ:

- the Z readout gives P(00) − P(11) = cos Φ;
- an X readout on the first qubit (`ry(-π/2)`) and a Y readout on the second (`rx(π/2)`) give ⟨X⊗Y⟩ = P(even) − P(odd) = −sin Φ.

Then Φ = atan2(sin Φ, cos Φ). We subtract the commanded 5θ and divide by 5 to get the error of one gate.

Why this probe:

- With both sin Φ and cos Φ the angle is never ambiguous while the total error stays within ±π, which is ±0.63 rad per gate. Our first probe read only P(11), which cannot tell an error d from π − d (Section 3.5).
- Depolarizing shrinks both readouts by the same factor, and the ratio inside atan2 cancels it.
- It assumes no error model. Each gate is corrected by its own measured error, whether that error is a fixed offset, grows with the angle, or depends on the sign.

Cost: 3 gates × 2 readouts × 256 shots = 1,536.

In [ ]:
def readout_check_circuits():
    """Prepare each of 000 to 111 with rx(pi) flips and nothing else."""
    circuits = {}
    for bits in TOFFOLI_INPUTS:
        qc = QuantumCircuit(3)
        for qubit, bit in enumerate(reversed(bits)):
            if bit == "1":
                qc.rx(np.pi, qubit)
        circuits[bits] = qc
    return circuits


def readout_matrices(counts_by_state):
    """Per-qubit 2x2 matrices A[read, prepared] estimated from the readout check."""
    matrices = []
    for qubit in range(3):
        a = np.zeros((2, 2))
        for bits, counts in counts_by_state.items():
            prepared = int(bits[2 - qubit])
            for outcome, n in counts.items():
                a[int(outcome[2 - qubit]), prepared] += n
        matrices.append(a / a.sum(axis=0, keepdims=True))
    return matrices


def correct_readout(counts, matrices, qubits=(0, 1, 2), clip=False):
    """Undo readout error on the given qubits by linear inversion; returns corrected weights that still sum to the shots.

    clip=True sets negative weights to 0 and renormalizes. That was our first version, kept only to show its bias.
    """
    n, shots = len(qubits), sum(counts.values())
    measured = np.zeros(2 ** n)
    for outcome, c in counts.items():
        measured[int("".join(outcome[2 - q] for q in reversed(qubits)), 2)] += c
    full = np.array([[1.0]])
    for q in reversed(qubits):
        full = np.kron(full, matrices[q])
    p = np.linalg.solve(full, measured / shots)
    if clip:
        p = np.clip(p, 0, None)
        p /= p.sum()
    return {format(i, f"0{n}b"): p[i] * shots for i in range(2 ** n)}


def resample(counts, rng, pseudo=0.5):
    """One bootstrap sample of 3-bit counts, drawn from (count + 0.5) over all 8 outcomes so unseen outcomes can appear."""
    outcomes = [format(i, "03b") for i in range(8)]
    weights = np.array([counts.get(k, 0) + pseudo for k in outcomes])
    return dict(zip(outcomes, rng.multinomial(int(round(sum(counts.values()))), weights / weights.sum())))


def full_matrix(check_counts):
    """The 8x8 matrix A[read, prepared] taken straight from the readout check, with no independence assumption."""
    a = np.zeros((8, 8))
    for bits, counts in check_counts.items():
        for outcome, n in counts.items():
            a[int(outcome, 2), int(bits, 2)] += n
    return a / a.sum(axis=0, keepdims=True)


def make_corrector(check_counts, full=False):
    """A function that undoes readout error on 3-bit counts, with per-qubit matrices or the full 8x8 matrix."""
    if not full:
        matrices = readout_matrices(check_counts)
        return lambda counts: correct_readout(counts, matrices)
    a = full_matrix(check_counts)

    def correct(counts):
        shots = sum(counts.values())
        measured = np.zeros(8)
        for outcome, c in counts.items():
            measured[int(outcome, 2)] += c
        p = np.linalg.solve(a, measured / shots)
        return {format(i, "03b"): p[i] * shots for i in range(8)}
    return correct


def marginal(weights, pair):
    """Sum 3-bit weights down to the two qubits of a pair."""
    out = {}
    for outcome, w in weights.items():
        key = "".join(outcome[2 - q] for q in reversed(pair))
        out[key] = out.get(key, 0) + w
    return out


def corrected_range(counts_by_test, tests, check_counts, full=False, repeats=200, seed=0):
    """Readout-corrected fidelity range, with 1 sigma errors from resampling the tests and the readout check together."""
    rng = np.random.default_rng(seed)
    with np.errstate(invalid="ignore"):  # fidelity_range's own error formula does not apply to corrected weights
        correct = make_corrector(check_counts, full)
        result = fidelity_range({key: correct(c) for key, c in counts_by_test.items()}, tests)
        samples = []
        for _ in range(repeats):
            correct = make_corrector({bits: resample(c, rng) for bits, c in check_counts.items()}, full)
            samples.append(fidelity_range({key: correct(resample(c, rng)) for key, c in counts_by_test.items()}, tests))
    for name in TEST_SETS:
        result["F_err"][name] = float(np.std([s["F"][name] for s in samples], ddof=1))
    result["S_TT_err"] = result["F_err"]["T"]
    result["F_LB_err"] = float(np.std([s["F_LB"] for s in samples], ddof=1))
    result["upper_err"] = float(np.std([s["upper"] for s in samples], ddof=1))
    return result


PROBE_REPEATS, PROBE_SHOTS = 5, 256


def probe_circuit(pair, theta, readout):
    """rxx(theta) repeated on |00> of a pair, then the Z readout or the X (first qubit) and Y (second qubit) readout."""
    qc = QuantumCircuit(3)
    for _ in range(PROBE_REPEATS):
        qc.rxx(theta, *pair)
    if readout == "XY":
        qc.ry(-np.pi / 2, pair[0])  # measure X
        qc.rx(np.pi / 2, pair[1])   # measure Y
    return qc


def probe_result(pair, theta, z_counts, xy_counts, correct):
    """(error of one rxx(theta) in radians, contrast) from its two readout-corrected probe readouts.

    The contrast, sqrt(cos^2 + sin^2), is 1 when the error is purely coherent and falls with random error.
    """
    z = marginal(correct(z_counts), pair)
    xy = marginal(correct(xy_counts), pair)
    cos_phi = (z.get("00", 0) - z.get("11", 0)) / sum(z.values())
    sin_phi = -(xy.get("00", 0) + xy.get("11", 0) - xy.get("01", 0) - xy.get("10", 0)) / sum(xy.values())
    total = np.angle(np.exp(1j * (np.arctan2(sin_phi, cos_phi) - PROBE_REPEATS * theta)))  # wrapped to -pi..pi
    return float(total / PROBE_REPEATS), float(np.hypot(cos_phi, sin_phi))


# Noise-free check of the two readouts
for phi in (0.3, 2.5, -2.0):
    z = Statevector(probe_circuit((0, 1), phi / PROBE_REPEATS, "Z")).probabilities_dict(qargs=[0, 1])
    xy = Statevector(probe_circuit((0, 1), phi / PROBE_REPEATS, "XY")).probabilities_dict(qargs=[0, 1])
    print(f"phi {phi:+.1f}: cos {np.cos(phi):+.3f}, P00 - P11 {z.get('00', 0) - z.get('11', 0):+.3f} | "
          f"-sin {-np.sin(phi):+.3f}, <XY> {xy.get('00', 0) + xy.get('11', 0) - xy.get('01', 0) - xy.get('10', 0):+.3f}")

In [ ]:
def readout_check_counts(model, seed, shots=128):
    """Simulate one readout check (8 circuits) on a noise model."""
    sim = AerSimulator(noise_model=MODELS[model][1], seed_simulator=seed)
    check = readout_check_circuits()
    batch = [qc.copy() for qc in check.values()]
    for qc in batch:
        qc.measure_all()
    result = sim.run(batch, shots=shots).result()
    return {bits: result.get_counts(i) for i, bits in enumerate(check)}


def pool(*checks):
    """Add readout checks from several rounds, state by state."""
    pooled = {}
    for check in checks:
        for bits, counts in check.items():
            for outcome, n in counts.items():
                pooled.setdefault(bits, {})
                pooled[bits][outcome] = pooled[bits].get(outcome, 0) + n
    return pooled


def rehearse(model, seed):
    """Simulate the round 1 readout check and the gate probes; return the measured error of each D0 gate."""
    change, noise, _ = MODELS[model]
    correct = make_corrector(readout_check_counts(model, seed))
    keys, batch = [], []
    for pair, theta in d0_gates:
        for readout in ("Z", "XY"):
            qc = change(probe_circuit(pair, theta, readout))
            qc.measure_all()
            keys.append((pair, theta, readout))
            batch.append(qc)
    result = AerSimulator(noise_model=noise, seed_simulator=seed + 50_000).run(batch, shots=PROBE_SHOTS).result()
    counts = {key: result.get_counts(i) for i, key in enumerate(keys)}
    return {(pair, theta): probe_result(pair, theta, counts[(pair, theta, "Z")], counts[(pair, theta, "XY")], correct)[0]
            for pair, theta in d0_gates}

The next cell rehearses the probe 30 times on four models and compares each measured gate error with the truth. It then builds D2 from the first rehearsal and compares its true fidelity with D0 and with D2 built from the exact errors.

In [ ]:
for model in ("disclosed", "angle", "large offset", "same direction"):
    runs = [rehearse(model, seed) for seed in range(30)]
    print(f"{model} model, 30 rehearsals")
    for gate in d0_gates:
        values = np.array([run[gate] for run in runs])
        print(f"  rxx({gate[1] / np.pi:+.2f}pi) on {gate[0]}: measured {values.mean():+.3f} ± {values.std(ddof=1):.3f}, "
              f"truth {TRUE_ERROR[model](gate[1]):+.3f}")
    exact = correct_each(designs["D0"], {gate: TRUE_ERROR[model](gate[1]) for gate in d0_gates})
    print(f"  true fidelity: D0 {true_fidelity(designs['D0'], model):.3f}, D2 from one rehearsal "
          f"{true_fidelity(correct_each(designs['D0'], runs[0]), model):.3f}, D2 from the exact errors {true_fidelity(exact, model):.3f}")

Does the readout-corrected range contain the true gate fidelity? Each case below is run 30 times with 128 shots per test circuit. The readout check is pooled from both rounds (8 × 256 shots in all). "inside" counts the runs where the true fidelity lies between F_LB − 2σ and upper + 2σ. The last column is the corrected upper end with clipping, our first version.

In [ ]:
for model in ("disclosed", "angle"):
    candidates = {"D0": designs["D0"], "D1c": designs["D1c"], "D2": correct_each(designs["D0"], rehearse(model, 0))}
    print(f"{model} model")
    for name, candidate in candidates.items():
        truth = true_fidelity(candidate, model)
        raw_rows, rows, clipped_upper = [], [], []
        for s in range(30):
            raw, counts, tests = simulate(candidate, model, seed=1000 + s)
            check = pool(readout_check_counts(model, 2000 + s), readout_check_counts(model, 3000 + s))
            raw_rows.append(raw)
            rows.append(corrected_range(counts, tests, check, seed=s))
            clipped = {key: correct_readout(c, readout_matrices(check), clip=True) for key, c in counts.items()}
            clipped_upper.append(fidelity_range(clipped, tests)["upper"])
        lb, ub = np.array([r["F_LB"] for r in rows]), np.array([r["upper"] for r in rows])
        lb_bar, ub_bar = np.mean([r["F_LB_err"] for r in rows]), np.mean([r["upper_err"] for r in rows])
        inside = np.mean([r["F_LB"] - 2 * r["F_LB_err"] <= truth <= r["upper"] + 2 * r["upper_err"] for r in rows])
        print(f"  {name:3s} true {truth:.3f} | raw [{np.mean([r['F_LB'] for r in raw_rows]):.3f}, {np.mean([r['upper'] for r in raw_rows]):.3f}] | "
              f"corrected [{lb.mean():.3f}, {ub.mean():.3f}], F_LB spread {lb.std(ddof=1):.3f} vs bar {lb_bar:.3f}, "
              f"upper spread {ub.std(ddof=1):.3f} vs bar {ub_bar:.3f} | inside {inside:.0%} | clipped upper {np.mean(clipped_upper):.3f}")

### 3.5 What Section 3 found, and what we changed

What the practice runs show:

- The true fidelities follow the physics. With a fixed over-rotation (disclosed model), D2* is far ahead: 0.990 against 0.933 for D1c and 0.929 for D0. When errors grow with the angle (angle model), D1c is ahead: 0.921 against 0.913 for D2* and 0.901 for D0 (3.1).
- The error bars are right. Over 200 runs the real spread of F_LB matches the formula: 0.0185 against 0.0184 in the disclosed model, 0.0201 against 0.0197 in the angle model (3.2).
- One run at 128 shots can see a calibration gain as large as the disclosed model's (D2* − D0 = +0.125, 2σ = 0.055). It cannot see the D1c − D0 gaps: +0.012 and +0.048, against 2σ of 0.055 and 0.067 (3.3). Round 2 cuts 2σ by √2, to about 0.047 in the angle model, so a gap like +0.048 would sit right at the edge.
- Readout alone takes a perfect gate's raw range to [0.756, 0.912] (3.1). The raw range is a floor for the whole run, not the gate's own number.
- The gate probe measures each of D0's three gates within 0.004 rad of the truth, with a spread of 0.009 to 0.017 rad, in four models, including a 0.35 rad error and an error that follows the sign of the angle. D2 built from one rehearsal reaches 0.989 against 0.990 with the exact errors, or matches the exact value (0.913) in the angle model (3.4).
- The readout-corrected range contains the true gate fidelity within 2σ in 90 to 100% of runs, and its error bars match its real spread (F_LB 0.031 to 0.033 against 0.030 to 0.032) (3.4). Its upper end sits up to 0.011 below the truth in the disclosed model, because the smallest of three noisy pass rates is pulled down by their noise (by about 0.85σ for three equal rates). So we read it with its error bars.

What we changed, and why:

1. We first planned to rank designs on the truth table. In the starter kit's model it ranks D0 above D1c while D1c is the better gate (Decision 1), so we moved to the fidelity range.
2. Our first gate probe repeated `rxx(π/2)` 3 times and `rxx(π/4)` 2 times, read only P(11), and fitted one fixed offset and one scale factor to all gates. In the disclosed model it found the 0.2 rad offset only to ±0.073 rad (earlier draft, not kept).
3. The second probe repeated the gates 5 and 6 times, which made the error larger and gave ±0.026 rad. But P(11) measures only the sine of the error, so an error d and π − d look the same, and past 0.26 rad per gate it gives the wrong one. A cheap single-gate probe (64 shots) added to pick the right answer made things worse: 0.227 ± 0.047 for a true 0.2, and 0.259 for a true 0.35, because 64 shots cannot tell 0.20 from 0.32 (earlier draft, not kept).
4. The probe in 3.4 reads both the sine and the cosine, so the angle is clear up to ±0.63 rad per gate, and it measures each gate of D0 directly instead of fitting a model.
5. Our first readout correction set negative weights to 0 and renormalized. That pulled every pass rate down: the corrected upper end for D2 was 0.963 against a true 0.989 (last column in 3.4). Linear inversion without clipping removes this bias.
6. Our first bootstrap redrew only outcomes that had already appeared, so its error bars came out 10 to 15% smaller than the real spread. Adding 0.5 to every outcome before redrawing fixed this.

## 4. Hardware plan and tools

This section gets everything ready for the lab session (4:00 PM to midnight). Nothing here spends shots: 4.3 only asks the relay for our budget, and `submit_batch` sends nothing to the lab unless `SUBMIT = True`. Section 4.7 runs the whole session on a local stand-in for the lab, so the code that reads the lab's files is tested before real data arrives. The starter notebook asks for this: "Do not use hardware to debug a circuit."

### 4.1 Hypotheses, written before any hardware data

Written at 14:13 on 24 October. The starter notebook says none of its simulators is "a calibration model of the live device", so our noise models are guesses. Each hypothesis below names the data that tests it and what it would change. Together they answer the Section 10 question "What would change your mind?"

| | Hypothesis | Test | If true | If false |
|---|---|---|---|---|
| H1 | Each `rxx` turns too far by the same fixed amount, like the disclosed model's +0.2 rad | the three measured gate errors have the same sign and agree within 2σ | D2 should beat D0 by more than 2σ of one round, about 0.055 in raw F_LB (Section 3.3) | D2 ties D0, and D0 against D1c is the main question |
| H2 | The error grows with the size of the angle | the −π/4 gate's error has the opposite sign of the +π/2 gates' error and about half its size (ratio near −0.5) | D1c, with smaller angles, should tie or beat D0 | smaller angles buy nothing |
| H3 | Different ion pairs have different errors | (0, 1) and (0, 2), both at +π/2, differ by more than 2σ of the difference (about 0.035 rad) | the per-gate correction in D2 already handles it | one correction for all pairs would have been enough |
| H4 | Readout reads 1 as 0 more often than 0 as 1 | P(0 given 1) > P(1 given 0) on each qubit in the readout check | same pattern as the disclosed model | the correction handles either case |
| H5 | Readout errors are independent between qubits | the independence test in 4.6 gives p ≥ 0.05 | per-qubit matrices | the full 8×8 matrix (decided now) |
| H6 | Random gate error is small next to the coherent part | the probe contrast stays near 1 after 5 gates | correcting the angles gives most of the possible gain | D2 gains less, and D1c may win |
| H7 | The device stays calibrated through the evening | one combined test of the 9 changes from round 1 to round 2 (6 readout rates, 3 gate errors) gives p ≥ 0.05 | pool the two rounds | report the rounds separately; D2's correction may be stale |

Session plan:

| Batch | When | What | Circuits | Shots |
|---|---|---|---|---|
| `round1-A` | 16:00 | readout check (8 × 128) and gate probes (6 × 256) | 14 | 2,560 |
| `round1-B` | when `round1-A` is back; if it takes over 30 minutes, D0 and D1c go at once as `round1-B` and D2 follows as `round1-C` | D0, D1c and D2, 24 tests each × 128, rotated order | 72 | 9,216 |
| `round2` | when round 1 is back, sent by 23:00 at the latest | readout check, gate probes at 128, and the top two designs × 24 × 128 | 62 | 7,936 |
| spare | | reruns | | 288 |

One teammate sends every batch, from one laptop. If the lab refuses the empty circuit that prepares 000, we send `rz(0)` in its place: `rz` runs in software and adds no error.

### 4.2 Build every circuit and check it before the lab sees it

`preflight` checks each job: the circuit passes `validate_submission_circuit`, the shot number is on the lab's menu, the circuit survives the trip to OpenQASM and back unchanged, and its ideal outcome is right. D2 is checked separately in 4.6, because its angles are shifted on purpose and it is not an exact Toffoli on an ideal machine.

In [ ]:
import tempfile
from collections import Counter

from toffoli_challenge.validation import STANDARD_SHOTS


def job(circuit, shots, **info):
    return {"circuit": circuit, "shots": shots, "info": info}


def readout_jobs(shots=128):
    """The 8 readout-check circuits."""
    return [job(qc, shots, kind="readout", prepared=bits) for bits, qc in readout_check_circuits().items()]


def probe_jobs(shots):
    """The Z and X-Y probes of each of D0's three rxx gates."""
    return [job(probe_circuit(pair, theta, readout), shots, kind="probe", pair=list(pair), theta=theta, readout=readout)
            for pair, theta in d0_gates for readout in ("Z", "XY")]


def test_jobs(candidates, shots=128):
    """All 24 tests of each design, in rotated order so no design always runs first."""
    tests = {name: test_circuits(circuit) for name, circuit in candidates.items()}
    names = list(candidates)
    jobs = []
    for i, key in enumerate(tests[names[0]]):
        for name in names[i % len(names):] + names[:i % len(names)]:
            circuit, expected = tests[name][key]
            jobs.append(job(circuit, shots, kind="test", design=name, test_set=key[0], labels=key[1], expected=expected))
    return jobs


def preflight(jobs, exact_designs=("D0", "D1c")):
    """Check every job before it is sent; raise on the first problem."""
    for j in jobs:
        circuit, info = j["circuit"], j["info"]
        validate_submission_circuit(circuit)
        assert j["shots"] in STANDARD_SHOTS, j["shots"]
        back = qasm2.loads(qasm2.dumps(circuit), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)
        assert Operator(back).equiv(Operator(circuit)), f"the QASM round trip changed a circuit: {info}"
        ideal = Statevector(circuit).probabilities_dict()
        if info["kind"] == "readout":
            assert ideal.get(info["prepared"], 0) > 1 - 1e-9, info
        if info["kind"] == "test" and info["design"] in exact_designs:
            assert ideal.get(info["expected"], 0) > 1 - 1e-9, info
    # a perfect gate must give probe error 0 and contrast 1
    for pair, theta in {(tuple(j["info"]["pair"]), j["info"]["theta"]) for j in jobs if j["info"]["kind"] == "probe"}:
        error, contrast = probe_result(pair, theta, exact_counts(probe_circuit(pair, theta, "Z")),
                                       exact_counts(probe_circuit(pair, theta, "XY")), lambda c: c)
        assert abs(error) < 1e-9 and abs(contrast - 1) < 1e-9, (pair, theta, error, contrast)
    kinds = dict(Counter(j["info"]["kind"] for j in jobs))
    print(f"{len(jobs)} circuits passed {kinds}, {sum(j['shots'] for j in jobs):,} shots")


preflight(readout_jobs(128) + probe_jobs(256))
preflight(test_jobs({"D0": designs["D0"], "D1c": designs["D1c"]}))

plan = {
    "round1-A": readout_jobs(128) + probe_jobs(256),
    "round1-B": test_jobs({"D0": designs["D0"], "D1c": designs["D1c"], "D2": designs["D0"]}),  # D0 stands in for D2 to count shots
    "round2": readout_jobs(128) + probe_jobs(128) + test_jobs({"first": designs["D0"], "second": designs["D1c"]}),
}
for batch, jobs in plan.items():
    print(f"{batch:9s} {len(jobs):3d} circuits {sum(j['shots'] for j in jobs):6,d} shots")
used = sum(j["shots"] for jobs in plan.values() for j in jobs)
assert used <= 20_000
print(f"total {used:,} shots, spare {20_000 - used:,}")

### 4.3 Connection check

`Lab` wraps the starter kit's relay functions. `connect_team` only stores the team values in memory. The status call asks the relay for our budget and the queue window, which costs nothing and works at any time.

In [ ]:
from toffoli_challenge import (
    bitstrings_to_counts,
    connect_team,
    fetch_my_results,
    remaining_hardware_shots,
    results_by_queue_id,
    submit_circuit,
)


class Lab:
    """The real lab, through the starter kit's relay functions."""
    log_path = QUEUE_LOG
    results_dir = RESULTS_DIR

    def __init__(self):
        connect_team(os.environ["TEAM_ID"], os.environ["TEAM_SECRET"])

    def status(self):
        return remaining_hardware_shots()

    def remaining(self):
        return self.status()["hardware_shots_remaining"]

    def submit(self, circuit, shots):
        return submit_circuit(circuit, shots=shots)

    def fetch(self):
        return fetch_my_results(destination=self.results_dir)


LAB = None
if os.environ.get("TEAM_ID") and os.environ.get("TEAM_SECRET"):
    try:
        LAB = Lab()
        status = LAB.status()
        window = status.get("submission_window", {})
        print(f"charged {status['charged_hardware_shots']}, remaining {status['hardware_shots_remaining']}; "
              f"queue {'OPEN' if window.get('open') else 'CLOSED'} (opens {window.get('opens_at')}, closes {window.get('closes_at')})")
    except Exception as error:  # bad team values, no network, or relay down: the rest of the notebook still runs
        print("real lab not connected:", error)
else:
    print("TEAM_ID or TEAM_SECRET not set: the real lab is not connected")

### 4.4 Sending a batch

`submit_batch` sends one batch to the real lab only when `SUBMIT = True` and the operator then types the batch name when asked. The typed name is a second lock: a teammate who runs the whole notebook with a `SUBMIT = True` left in a cell is asked for a name and sends nothing. It writes each queue ID to the log right after that circuit is sent, the same habit as IQM's "retrieve results later" example: save the ID first, fetch whenever. It refuses a batch name that is already in the log, so running a cell twice cannot spend the shots twice. It checks the budget before sending. It stops at the first error and never retries by itself, because a relay timeout (30 s) does not prove the circuit was not queued: check the remaining shots and the log first.

In [ ]:
SUBMIT = False  # the operator sets this to True for one batch, then back to False


def logged(lab):
    return json.loads(lab.log_path.read_text()) if lab.log_path.exists() else []


def submit_batch(batch, jobs, lab):
    """Send each job and log its queue ID straight after it is sent."""
    shots = sum(j["shots"] for j in jobs)
    if any(entry["batch"] == batch for entry in logged(lab)):
        raise RuntimeError(f"batch {batch!r} is already in {lab.log_path.name}, so it was sent before. "
                           "To send only the circuits a stopped batch missed, give them a new batch name.")
    if isinstance(lab, Lab):
        if not SUBMIT:
            print(f"SUBMIT is False: {batch!r} not sent ({len(jobs)} circuits, {shots:,} shots)")
            return []
        if input(f"Type {batch} to send {len(jobs)} circuits ({shots:,} shots) to the lab: ").strip() != batch:
            print(f"{batch!r} not sent")
            return []
    remaining = lab.remaining()
    if shots > remaining:
        raise RuntimeError(f"{batch!r} needs {shots:,} shots but only {remaining:,} remain")
    entries = []
    for n, j in enumerate(jobs, start=1):
        try:
            receipt = lab.submit(j["circuit"], j["shots"])
        except Exception:
            print(f"stopped at circuit {n} of {len(jobs)}; {len(entries)} were sent and logged. "
                  "Do not resend: check the remaining shots and the queue log first.")
            raise
        entries.append(log_submission(receipt, log_path=lab.log_path, batch=batch, shots=j["shots"],
                                      qasm=qasm2.dumps(j["circuit"]), relay_receipt=receipt["submission"], **j["info"]))
    print(f"{batch}: sent {len(entries)} circuits, {shots:,} shots, queue IDs {entries[0]['queue_id']} to {entries[-1]['queue_id']}")
    return entries

### 4.5 Collecting results

`collect` fetches every file returned so far, keeps the ones whose queue IDs belong to the batch, and lists what is still pending. The starter kit's `results_by_queue_id` also checks that each file holds the number of shots its name says. Once a batch is complete it prints how long the lab took, which decides the round 1 order (4.1).

In [ ]:
def collect(batch, lab):
    """Return ([(log entry, counts)], pending queue IDs) for one batch."""
    entries = [entry for entry in logged(lab) if entry["batch"] == batch]
    returned = results_by_queue_id(lab.fetch())
    done = [(entry, bitstrings_to_counts(returned[entry["queue_id"]])) for entry in entries if entry["queue_id"] in returned]
    pending = [entry["queue_id"] for entry in entries if entry["queue_id"] not in returned]
    for entry, counts in done:
        assert sum(counts.values()) == entry["shots"], entry["queue_id"]
    message = f"{batch}: {len(done)} of {len(entries)} returned"
    if pending:
        message += f"; pending queue IDs {pending}"
    elif entries:
        minutes = (datetime.now() - datetime.fromisoformat(entries[-1]["logged_at"])).total_seconds() / 60
        message += f"; complete, {minutes:.1f} minutes after the last circuit was sent"
    print(message)
    return done, pending

### 4.6 Turning lab files into numbers

- `independence_test` checks H5. It compares the 8×8 readout table with what the per-qubit matrices predict, and gets the p-value by simulating many readout checks in which the qubits really are independent. p < 0.05 means the full 8×8 matrix (Decisions).
- `probe_table` gives each D0 gate's error and contrast, with 1σ errors from the same smoothed bootstrap as 3.4. `hypothesis_report` prints the numbers H1 to H6 need. For H2, the ratio of the −π/4 gate's error to the +π/2 gates' error is +1 for a fixed offset, −0.5 for an error that grows with the angle, and −1 for an error that follows the sign of the angle.
- `build_d2` corrects D0 with the measured errors, then checks that D2 is D0 with only its `rxx` angles changed, each by exactly the measured error, and that adding the errors back gives an exact Toffoli.
- `design_report` gives the raw and readout-corrected ranges of each design, ranks the designs on raw F_LB, and applies the 2σ rule (Decisions 3 and 4). Tests repeated in two rounds are pooled.
- `drift_report` checks H7 by comparing round 2 with round 1. It prints each of the 9 changes in σ, but the verdict comes from one combined test: the sum of the squared changes in σ, compared with a chi-square distribution with 9 degrees of freedom. Checking 9 numbers one at a time at 2σ would flag at least one about 34% of the time even with no drift (1 − 0.954⁹); the combined test keeps the false-alarm rate at 5%, the same as a single 2σ check; the rehearsal below shows two such false alarms at 2.2σ.

Two things to expect in the results. Readout-corrected numbers can come out above 1, because the linear correction is unbiased but noisy; we do not clip them, since clipping biases them down (Section 3.5). And the probe contrast has an error of about 0.05 at 256 shots, so H6 can only see random error that pulls the contrast after 5 gates below about 0.9.

In [ ]:
def readout_counts(done):
    """Readout-check counts by prepared state; a rerun of a state adds to its counts."""
    out = {}
    for entry, counts in done:
        if entry["kind"] == "readout":
            out[entry["prepared"]] = merged(out.get(entry["prepared"], {}), counts)
    return out


def independence_test(check_counts, samples=2000, seed=0):
    """H5: p-value for "readout errors are independent between qubits"."""
    rng = np.random.default_rng(seed)
    outcomes = [format(i, "03b") for i in range(8)]

    def predicted(counts_by_state):
        matrices = readout_matrices(counts_by_state)
        return {bits: np.array([np.prod([matrices[q][int(o[2 - q]), int(bits[2 - q])] for q in range(3)]) for o in outcomes])
                for bits in counts_by_state}

    def statistic(counts_by_state):
        total = 0.0
        for bits, p in predicted(counts_by_state).items():
            counts = counts_by_state[bits]
            shots = sum(counts.values())
            observed = np.array([counts.get(o, 0) for o in outcomes])
            total += np.sum((observed - shots * p) ** 2 / np.maximum(shots * p, 0.5))
        return total

    model = predicted(check_counts)
    null = []
    for _ in range(samples):
        fake = {bits: dict(zip(outcomes, rng.multinomial(sum(counts.values()), model[bits] / model[bits].sum())))
                for bits, counts in check_counts.items()}
        null.append(statistic(fake))
    return float(np.mean(np.array(null) >= statistic(check_counts)))


def probe_table(done, check_counts, full=False, repeats=200, seed=0):
    """Each D0 gate's error and contrast from the probe results, with bootstrap 1 sigma errors."""
    rng = np.random.default_rng(seed)
    counts = {}
    for e, c in done:
        if e["kind"] == "probe":
            key = (tuple(e["pair"]), e["theta"], e["readout"])
            counts[key] = merged(counts.get(key, {}), c)
    correct = make_corrector(check_counts, full)
    table = {}
    for pair, theta in d0_gates:
        z, xy = counts[(pair, theta, "Z")], counts[(pair, theta, "XY")]
        error, contrast = probe_result(pair, theta, z, xy, correct)
        boot = np.array([probe_result(pair, theta, resample(z, rng), resample(xy, rng),
                                      make_corrector({b: resample(c, rng) for b, c in check_counts.items()}, full))
                         for _ in range(repeats)])
        table[(pair, theta)] = {"error": error, "error_err": float(boot[:, 0].std(ddof=1)),
                                "contrast": contrast, "contrast_err": float(boot[:, 1].std(ddof=1))}
    return table


def hypothesis_report(check_counts, table, p_independent):
    matrices = readout_matrices(check_counts)
    print("H4 readout: P(read 1 given 0), P(read 0 given 1)")
    for q in range(3):
        n0 = sum(sum(c.values()) for bits, c in check_counts.items() if bits[2 - q] == "0")
        n1 = sum(sum(c.values()) for bits, c in check_counts.items() if bits[2 - q] == "1")
        e01, e10 = matrices[q][1, 0], matrices[q][0, 1]
        print(f"  q{q}: {e01:.3f} ± {np.sqrt(e01 * (1 - e01) / n0):.3f}, {e10:.3f} ± {np.sqrt(e10 * (1 - e10) / n1):.3f}")
    print(f"H5 independence test: p = {p_independent:.3f}, so we use the {'full 8x8 matrix' if p_independent < 0.05 else 'per-qubit matrices'}")
    print("H1, H2, H3, H6 gate probes: error per gate (rad), contrast after 5 gates, contrast per gate")
    for (pair, theta), row in table.items():
        print(f"  rxx({theta / np.pi:+.2f}pi) on {pair}: error {row['error']:+.3f} ± {row['error_err']:.3f}, "
              f"contrast {row['contrast']:.3f} ± {row['contrast_err']:.3f}, per gate {row['contrast'] ** (1 / PROBE_REPEATS):.4f}")
    a, b, c = (table[gate] for gate in d0_gates)  # +pi/2 on (0, 1), +pi/2 on (0, 2), -pi/4 on (1, 2)
    gap, gap_err = a["error"] - b["error"], np.hypot(a["error_err"], b["error_err"])
    print(f"  H3: (0, 1) minus (0, 2) = {gap:+.3f} ± {gap_err:.3f}, {abs(gap) / gap_err:.1f} sigma")
    mean = (a["error"] + b["error"]) / 2
    mean_err = np.hypot(a["error_err"], b["error_err"]) / 2
    if abs(mean) > 2 * mean_err:
        ratio = c["error"] / mean
        ratio_err = abs(ratio) * np.hypot(c["error_err"] / c["error"], mean_err / mean) if c["error"] else np.nan
        print(f"  H2: ratio of the -pi/4 error to the +pi/2 error = {ratio:+.2f} ± {ratio_err:.2f} "
              "(fixed offset +1, grows with angle -0.5, follows the sign -1)")
    else:
        print(f"  H1, H2: the +pi/2 gates' error {mean:+.3f} ± {mean_err:.3f} is within 2 sigma of 0, so the ratio is not defined")


def build_d2(table):
    """D0 with each rxx corrected by its measured error, after checking that nothing else changed."""
    errors = {gate: row["error"] for gate, row in table.items()}
    d0 = designs["D0"]
    d2 = correct_each(d0, errors)
    validate_submission_circuit(d2)
    assert len(d2.data) == len(d0.data)
    restore = {}
    for old, new in zip(d0.data, d2.data):
        assert old.operation.name == new.operation.name
        assert [d0.find_bit(q).index for q in old.qubits] == [d2.find_bit(q).index for q in new.qubits]
        if old.operation.name == "rxx":
            pair = tuple(sorted(d0.find_bit(q).index for q in old.qubits))
            theta, shifted = float(old.operation.params[0]), float(new.operation.params[0])
            error = errors[(pair, round(theta, 10))]
            assert abs(theta - shifted - error) < 1e-12
            restore[(pair, round(shifted, 10))] = -error
        else:
            assert list(map(float, old.operation.params)) == list(map(float, new.operation.params))
    _, restored_fidelity, _ = compare_to_toffoli(correct_each(d2, restore))
    assert abs(restored_fidelity - 1) < 1e-9
    print("D2 checked: only the rxx angles changed, and adding the measured errors back gives an exact Toffoli")
    return d2


def merged(*count_dicts):
    out = {}
    for counts in count_dicts:
        for outcome, n in counts.items():
            out[outcome] = out.get(outcome, 0) + n
    return out


def design_report(done, check_counts, names, full=False):
    """Raw and readout-corrected ranges of each design, ranked on raw F_LB with the 2 sigma rule."""
    rows = {}
    for name in names:
        counts, tests = {}, {}
        for entry, c in done:
            if entry["kind"] == "test" and entry["design"] == name:
                key = (entry["test_set"], entry["labels"])
                counts[key] = merged(counts.get(key, {}), c)
                tests[key] = (None, entry["expected"])
        assert len(counts) == 24, f"{name}: only {len(counts)} of 24 tests returned"
        rows[name] = (fidelity_range(counts, tests), corrected_range(counts, tests, check_counts, full=full))
        report(f"{name} raw", rows[name][0])
        report(f"{name} corr.", rows[name][1])
    ranked = sorted(rows, key=lambda n: rows[n][0]["F_LB"], reverse=True)
    best = rows[ranked[0]][0]
    for name in ranked[1:]:
        gap = best["F_LB"] - rows[name][0]["F_LB"]
        two_sigma = 2 * np.hypot(best["F_LB_err"], rows[name][0]["F_LB_err"])
        print(f"raw F_LB({ranked[0]}) - F_LB({name}) = {gap:+.3f}, 2 sigma = {two_sigma:.3f} -> {'different' if gap > two_sigma else 'tied'}")
    return rows, ranked


def drift_report(check_1, check_2, table_1, table_2):
    """H7: round 2 against round 1, in sigma, and one combined test of all 9 changes."""
    from scipy.stats import chi2

    z = []
    for q in range(3):
        for prepared, label in ((0, "P(read 1 given 0)"), (1, "P(read 0 given 1)")):
            rates = []
            for check in (check_1, check_2):
                n = sum(sum(c.values()) for bits, c in check.items() if bits[2 - q] == str(prepared))
                rate = readout_matrices(check)[q][1 - prepared, prepared]
                rates.append((rate, np.sqrt(max(rate * (1 - rate), 1 / n) / n)))
            (r1, s1), (r2, s2) = rates
            z.append((r2 - r1) / np.hypot(s1, s2))
            print(f"  q{q} {label}: {r1:.3f} then {r2:.3f}, change {abs(z[-1]):.1f} sigma")
    for gate in d0_gates:
        one, two = table_1[gate], table_2[gate]
        z.append((two["error"] - one["error"]) / np.hypot(one["error_err"], two["error_err"]))
        print(f"  rxx({gate[1] / np.pi:+.2f}pi) on {gate[0]}: error {one['error']:+.3f} then {two['error']:+.3f}, "
              f"change {abs(z[-1]):.1f} sigma")
    total = float(np.sum(np.square(z)))
    p = float(chi2.sf(total, df=len(z)))
    print(f"  combined: sum of squares {total:.1f} with {len(z)} degrees of freedom, p = {p:.3f} -> "
          f"{'drift: report the rounds separately' if p < 0.05 else 'no drift seen: pool the rounds'}")
    return p

### 4.7 Rehearsal of the whole session on a stand-in lab

`StandInLab` behaves like the lab from our side: it accepts the same circuits, charges shots, hands back the same receipts, and returns files named `<queueID>_<shots>_<teamID>.txt` holding a raw bitstring list. It simulates the disclosed noise model and keeps its log and files in a temporary folder, so the real queue log is never touched. The cells below run batch `round1-A`, the analysis and D2, batch `round1-B`, and `round2`, exactly as we will at 4 PM, but with `rehearsal` in place of `LAB`.

In [ ]:
class StandInLab:
    """A local stand-in for the lab, simulated with one of the Section 3 noise models."""

    def __init__(self, model="disclosed", budget=20_000, seed=0):
        folder = Path(tempfile.mkdtemp(prefix="standin_lab_"))
        self.log_path, self.results_dir = folder / "queue_log.json", folder / "returned_results"
        self.model, self.budget, self.seed = model, budget, seed
        self.files, self.next_id = {}, 1

    def remaining(self):
        return self.budget

    def submit(self, circuit, shots):
        validate_submission_circuit(circuit)
        if shots not in STANDARD_SHOTS or shots > self.budget:
            raise RuntimeError("shot number not on the menu, or over budget")
        change, noise, _ = MODELS[self.model]
        qc = change(circuit.copy())
        qc.measure_all()
        sim = AerSimulator(noise_model=noise, seed_simulator=self.seed + self.next_id)
        self.files[f"{self.next_id}_{shots}_standin.txt"] = str(sim.run(qc, shots=shots, memory=True).result().get_memory())
        self.budget -= shots
        self.next_id += 1
        return {"submission": {"queue_id": self.next_id - 1, "shots": shots}}

    def fetch(self):
        self.results_dir.mkdir(parents=True, exist_ok=True)
        for name, text in self.files.items():
            (self.results_dir / name).write_text(text)
        return dict(self.files)


rehearsal = StandInLab("disclosed", seed=7)
batch_a = readout_jobs(128) + probe_jobs(256)
preflight(batch_a)
submit_batch("round1-A", batch_a, rehearsal)
done_a, _ = collect("round1-A", rehearsal)
check_1 = readout_counts(done_a)
p_independent = independence_test(check_1)
FULL_READOUT = p_independent < 0.05
table_1 = probe_table(done_a, check_1, FULL_READOUT)
hypothesis_report(check_1, table_1, p_independent)
d2 = build_d2(table_1)
print(f"stand-in only, true fidelity: D0 {true_fidelity(designs['D0'], 'disclosed'):.3f}, D2 {true_fidelity(d2, 'disclosed'):.3f}")

In [ ]:
round1_designs = {"D0": designs["D0"], "D1c": designs["D1c"], "D2": d2}
batch_b = test_jobs(round1_designs)
preflight(batch_b)
submit_batch("round1-B", batch_b, rehearsal)
done_b, _ = collect("round1-B", rehearsal)
round1, ranked = design_report(done_b, check_1, round1_designs, FULL_READOUT)
top_two = ranked[:2]
print("top two for round 2:", top_two)

In [ ]:
batch_2 = readout_jobs(128) + probe_jobs(128) + test_jobs({name: round1_designs[name] for name in top_two})
preflight(batch_2, exact_designs=[n for n in top_two if n != "D2"])
submit_batch("round2", batch_2, rehearsal)
done_2, _ = collect("round2", rehearsal)
check_2 = readout_counts(done_2)
print("H7 drift, round 2 against round 1")
drift_report(check_1, check_2, table_1, probe_table(done_2, check_2, FULL_READOUT))
print("final comparison: both rounds pooled, readout check pooled (8 x 256)")
final, final_ranked = design_report(done_b + done_2, pool(check_1, check_2), top_two, FULL_READOUT)
for name in final_ranked:
    print(f"stand-in only, true fidelity of {name}: {true_fidelity(round1_designs[name], 'disclosed'):.3f}")
print(f"shots used {20_000 - rehearsal.remaining():,}, spare {rehearsal.remaining():,}")
try:
    submit_batch("round1-A", batch_a, rehearsal)
except RuntimeError as error:
    print("guard against sending twice:", error)

## 5. Round 1 on the lab

The operator runs this section from 4:00 PM, one cell at a time. Every cell is safe to rerun: fetching results is free, a batch already in the queue log is reported and not sent again, and nothing goes to the lab unless `SUBMIT = True` and the operator types the batch name. After a kernel restart, rerun the whole notebook: the queue log and the files in `returned_results/` rebuild every number.

1. 5.1: the relay must accept our team values and show the queue OPEN.
2. 5.2: send `round1-A`, the readout check and gate probes (14 circuits, 2,560 shots).
3. 5.3: rerun every few minutes until `round1-A` is complete. It then reports H1 to H6 and builds D2.
4. 5.4: send round 1's tests. If D2 is built, all three designs go as `round1-B`. If `round1-A` has been out for 30 minutes or more without coming back, D0 and D1c go as `round1-B` straight away (rule in 4.1).
5. 5.5: only in that second case, send D2's tests as `round1-C` once 5.3 has built D2.
6. 5.6: rerun until round 1 is complete. It ranks the designs on raw F_LB and names the top two for round 2.

After each batch, the operator shares `submissions/queue_log.json` with the team. It holds queue IDs and circuits, no secrets, and teammates need it to run 5.3 and 5.6 on their own laptops. Teammates never send.

| Problem | What to do |
|---|---|
| `submit_batch` stops with an error | Do not resend. Rerun 5.1 to compare the charged shots with the queue log. |
| the relay refuses the first circuit of `round1-A`, the empty circuit that prepares 000 | If 5.1 shows nothing was charged, set `EMPTY_FALLBACK = True` in 5.2 and run it again. |
| a result has not come back after about 2 hours | Use 5.7 to send only the missing circuits again, from the 288 spare shots. |

### 5.1 Connection and budget check

In [ ]:
if LAB is None:
    print("real lab not connected: TEAM_ID and TEAM_SECRET are not set (Section 0)")
else:
    try:
        status = LAB.status()
        window = status.get("submission_window", {})
        in_log = sum(entry["shots"] for entry in logged(LAB))
        print(f"relay accepted our team: charged {status['charged_hardware_shots']:,}, remaining {status['hardware_shots_remaining']:,}; "
              f"queue {'OPEN' if window.get('open') else 'CLOSED'} (opens {window.get('opens_at')}, closes {window.get('closes_at')})")
        print(f"queue log on this laptop: {len(logged(LAB))} circuits, {in_log:,} shots")
        if status["charged_hardware_shots"] != in_log:
            print("charged shots and the queue log differ: find out why before sending anything")
    except Exception as error:
        print("relay did not accept the request:", error)

### 5.2 Send `round1-A`

In [ ]:
EMPTY_FALLBACK = False  # True only if the relay refused the empty circuit that prepares 000


def live_readout_jobs(shots=128):
    """Readout-check jobs; with EMPTY_FALLBACK the 000 circuit gets rz(0), which runs in software and changes nothing."""
    jobs = readout_jobs(shots)
    if EMPTY_FALLBACK:
        for j in jobs:
            if j["info"]["prepared"] == "000":
                j["circuit"].rz(0.0, 0)
    return jobs


def send_once(batch, jobs):
    """Send a batch unless the queue log already has it."""
    if LAB is None:
        print("real lab not connected (4.3)")
    elif any(entry["batch"] == batch for entry in logged(LAB)):
        print(f"{batch}: already sent (in the queue log)")
    else:
        preflight(jobs)
        submit_batch(batch, jobs, LAB)


SUBMIT = False  # operator: set True, run this cell once, and type the batch name when asked
try:
    send_once("round1-A", live_readout_jobs(128) + probe_jobs(256))
finally:
    SUBMIT = False

### 5.3 Collect `round1-A`, test H1 to H6, build D2

The readout model is chosen here by the rule fixed in 4.1: per-qubit matrices unless the independence test gives p < 0.05.

In [ ]:
ROUND1_A = ("round1-A", "round1-A-rerun")
ROUND1_TESTS = ("round1-B", "round1-C", "round1-tests-rerun")
live = {}


def job_key(info):
    """What a circuit measures, so that a rerun counts as the same circuit."""
    if info["kind"] == "readout":
        return ("readout", info["prepared"])
    if info["kind"] == "probe":
        return ("probe", tuple(info["pair"]), info["theta"], info["readout"])
    return ("test", info["design"], info["test_set"], info["labels"])


def live_collect(batches):
    """Collect a batch and its reruns; returns (done, circuits still missing), or (None, None) if there is nothing to collect."""
    if LAB is None:
        print("real lab not connected (4.3)")
        return None, None
    entries = [entry for entry in logged(LAB) if entry["batch"] in batches]
    if not entries:
        print(f"{batches[0]}: not sent yet")
        return None, None
    done = []
    try:
        for batch in batches:
            if any(entry["batch"] == batch for entry in entries):
                done += collect(batch, LAB)[0]
    except Exception as error:
        print("could not fetch results:", error)
        return None, None
    missing = {job_key(entry) for entry in entries} - {job_key(entry) for entry, _ in done}
    return done, missing


done_a, missing_a = live_collect(ROUND1_A)
if done_a is not None and missing_a:
    print(f"{len(missing_a)} of round1-A's circuits are still out: rerun this cell later")
elif done_a is not None:
    live["check_1"] = readout_counts(done_a)
    live["p_independent"] = independence_test(live["check_1"])
    live["full"] = live["p_independent"] < 0.05
    live["table_1"] = probe_table(done_a, live["check_1"], live["full"])
    hypothesis_report(live["check_1"], live["table_1"], live["p_independent"])
    live["D2"] = build_d2(live["table_1"])
    live["designs"] = {"D0": designs["D0"], "D1c": designs["D1c"], "D2": live["D2"]}
    print("D2 as OpenQASM:")
    print(qasm2.dumps(live["D2"]))

### 5.4 Send round 1's tests (`round1-B`)

In [ ]:
def round1_order(limit_minutes=30):
    """'together' once D2 is built; 'split' if round1-A has been out for limit_minutes without D2; otherwise 'wait'."""
    sent_a = [entry for entry in logged(LAB) if entry["batch"] == "round1-A"] if LAB is not None else []
    if not sent_a:
        return "wait"
    if "D2" in live:
        return "together"
    minutes = (datetime.now() - datetime.fromisoformat(sent_a[-1]["logged_at"])).total_seconds() / 60
    print(f"round1-A was sent {minutes:.0f} minutes ago and D2 is not built yet")
    return "split" if minutes >= limit_minutes else "wait"


SUBMIT = False  # operator: set True, run this cell once, and type the batch name when asked
try:
    order = round1_order()
    print("round 1 order:", order)
    if order == "together":
        send_once("round1-B", test_jobs(live["designs"]))
    elif order == "split":
        send_once("round1-B", test_jobs({"D0": designs["D0"], "D1c": designs["D1c"]}))
    else:
        print("nothing sent: send round1-A first, or rerun 5.3 and then this cell")
finally:
    SUBMIT = False

### 5.5 Send D2's tests (`round1-C`), only if `round1-B` went without D2

In [ ]:
SUBMIT = False  # operator: set True, run this cell once, and type the batch name when asked
try:
    sent_b = {entry["design"] for entry in logged(LAB) if entry["batch"] == "round1-B"} if LAB is not None else set()
    if not sent_b:
        print("round1-B not sent yet")
    elif "D2" in sent_b:
        print("round1-C not needed: D2 went with round1-B")
    elif "D2" not in live:
        print("D2 is not built yet: rerun 5.3 once round1-A is back, then this cell")
    else:
        send_once("round1-C", test_jobs({"D2": live["D2"]}))
finally:
    SUBMIT = False

### 5.6 Collect round 1 and rank the designs

In [ ]:
done_tests, missing_tests = live_collect(ROUND1_TESTS)
if done_tests is not None and missing_tests:
    print(f"{len(missing_tests)} of round 1's test circuits are still out: rerun this cell later")
elif done_tests is not None and "check_1" not in live:
    print("run 5.3 first: the readout check from round1-A is needed")
elif done_tests is not None:
    names = [name for name in ("D0", "D1c", "D2") if any(entry["design"] == name for entry, _ in done_tests)]
    if "D2" not in names:
        print("D2's tests (round1-C) are not sent or not back yet")
    else:
        live["round1"], live["ranked"] = design_report(done_tests, live["check_1"], names, live["full"])
        live["top_two"] = live["ranked"][:2]
        print("top two for round 2:", live["top_two"])

### 5.7 Send missing circuits again (only after about 2 hours)

`resend_missing` rebuilds each missing circuit from the OpenQASM in the queue log and sends it under a new batch name. The analysis counts a rerun as more shots of the same circuit. Uncomment the line you need, set `SUBMIT = True`, and run the cell once.

In [ ]:
def resend_missing(batches, new_batch):
    """Send again every circuit of these batches whose result has not come back."""
    _, missing = live_collect(batches)
    if not missing:
        print("nothing missing")
        return
    jobs, seen = [], set()
    for entry in logged(LAB):
        key = job_key(entry)
        if entry["batch"] in batches and key in missing and key not in seen:
            seen.add(key)
            circuit = qasm2.loads(entry["qasm"], custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)
            info = {k: v for k, v in entry.items() if k not in ("queue_id", "logged_at", "batch", "shots", "qasm", "relay_receipt")}
            jobs.append(job(circuit, entry["shots"], **info))
    print(f"{len(jobs)} circuits to send again, {sum(j['shots'] for j in jobs):,} shots")
    send_once(new_batch, jobs)


SUBMIT = False  # operator: set True only for a resend
try:
    pass
    # resend_missing(("round1-A",), "round1-A-rerun")
    # resend_missing(("round1-B", "round1-C"), "round1-tests-rerun")
finally:
    SUBMIT = False